# Coupled 2-PDE System via Native Complex Formulation
## 1D Wave/Advection System with Cross-Coupling

This notebook demonstrates how to solve a coupled system of 2 linear PDEs by packing them into a single complex PDE, using the special condition $c = -b$ and $d = a$.

---

## 1. The Governing Equation & Equivalence

**The Real System (2 PDEs):**
$$
\begin{align}
\partial_t u &= a\,\partial_x u + b\,\partial_x v + \nu \nabla^2 u \\
\partial_t v &= -b\,\partial_x u + a\,\partial_x v + \nu \nabla^2 v
\end{align}
$$
*Note: We added a small diffusion term $\nu \nabla^2$ to prevent high-frequency numerical blow-up (explained in Section 3).*

**The Complex Form (1 PDE):**
By defining $A = u + iv$, the system collapses beautifully into a single complex advection-diffusion equation:
$$
\partial_t A = (a - ib)\,\partial_x A + \nu \nabla^2 A
$$

---

## 2. Reformulation for the Solver

In Fourier space ($\partial_x \to i\xi$, $\nabla^2 \to -(\xi^2 + \eta^2)$):

$$
\text{Linear symbol:} \quad i(a - ib)\xi - \nu(\xi^2 + \eta^2) = (b + ia)\xi - \nu(\xi^2 + \eta^2)
$$

The equation in the solver's format:

$$
\partial_t A = \text{psiOp}\left( (b + ia)\xi - \nu(\xi^2 + \eta^2), A \right)
$$

---

## 3. Why the Diffusion Term ($\nu$) is Mandatory

If we set $\nu = 0$, the Fourier symbol is purely $(b + ia)\xi$. 
The **real part** of this symbol is $b\xi$. 
* For $\xi > 0$, the modes grow exponentially as $e^{b\xi t}$.
* For $\xi < 0$, the modes decay exponentially.

This asymmetric exponential growth is a hallmark of the continuous 1D wave equation when written in this specific complex form, and it will instantly crash any numerical solver due to floating-point overflow. Adding $\nu \nabla^2 A$ (which contributes $-\nu \xi^2$ to the real part) acts as a low-pass filter, damping high frequencies and stabilizing the simulation while preserving the wave dynamics.

# Implementation
## 0. Imports

In [ ]:
from solver import PDESolver, psiOp
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML

## 1. Physical and simulation parameters

In [ ]:
# ── Coupling Coefficients ──
A_COEFF = 0.5   # 'a' in the math (symmetric advection)
B_COEFF = 2.0   # 'b' in the math (cross-coupling / wave speed)
NU = 0.2        # Numerical dissipation (mandatory for stability)

# ── Grid and Time ──
# We use a 2D grid to satisfy the PDESolver signature, but the physics is 1D (invariant in y)
Lx, Ly = 20.0, 10.0   
Nx, Ny = 256, 128       # High resolution in X, minimal in Y

Lt, Nt = 20.0, 2000   
n_frames = 100        

## 2. Grid setup

In [ ]:
xs_1d = np.linspace(-Lx/2, Lx/2, Nx)
ys_1d = np.linspace(-Ly/2, Ly/2, Ny)

# CRITICAL FIX: psipy internally uses indexing='ij' (axis 0 = x, axis 1 = y).
xx, yy = np.meshgrid(xs_1d, ys_1d, indexing='ij')

## 3. SymPy symbols and principal symbol

In [ ]:
t, x, y = sp.symbols('t x y', real=True)
xi, eta = sp.symbols('xi eta', real=True)

# Define A as a complex-valued field
A_field = sp.Function('A')(t, x, y)

# ── Linear symbol in Fourier space ──
# From: ∂A/∂t = (a - i*b)∂A/∂x + ν∇²A
# Fourier: ∂x → iξ, ∇² → -(ξ² + η²)
# So: i(a - i*b)ξ - ν(ξ² + η²) = (b + i*a)ξ - ν(ξ² + η²)

k2 = xi**2 + eta**2
symbol_linear = (B_COEFF + sp.I * A_COEFF) * xi - NU * k2

print('Principal symbol (linear part):')
print('  a(ξ, η) = ', symbol_linear)

## 4. The Complexified PDE

In [ ]:
# ∂A/∂t = psiOp( (b + i*a)ξ - ν(ξ² + η²), A )

equation = sp.Eq(
    sp.diff(A_field, t),
    psiOp(symbol_linear, A_field)
)

print('Complexified Coupled System:')
print('  ∂A/∂t = psiOp( (b + i*a)ξ - ν(ξ² + η²), A )')
print(f'\nParameters: a={A_COEFF}, b={B_COEFF}, ν={NU}')

## 5. Initial conditions

In [ ]:
def initial_condition_coupled(xx, yy):
    """
    We start with a Gaussian pulse in 'u' (real part) and zero in 'v' (imaginary part).
    Because of the cross-coupling (b != 0), this pulse will split into two waves
    traveling in opposite directions, while exchanging energy between u and v.
    """
    # u(x,0) = Gaussian, v(x,0) = 0
    u = np.exp(-(xx + 0)**2 / 3.0)
    v = np.zeros_like(xx)
    
    return u + 1j * v

print("Using: Gaussian pulse in 'u', zero in 'v'")

## 6. Solver setup

In [ ]:
solver = PDESolver(equation)

solver.setup(
    Lx=Lx, Ly=Ly,
    Nx=Nx, Ny=Ny,
    Lt=Lt, Nt=Nt,
    boundary_condition='periodic',
    initial_condition=initial_condition_coupled,
    n_frames=n_frames,
    plot=True,
)

## 7. Solve

In [ ]:
frames = solver.solve()

## 8. Visualization

In [ ]:
plt.rcParams['animation.embed_limit'] = 2**128

# Visualize the 'real' component, which corresponds to 'u' in the original 2-PDE system
ani = solver.animate(
    component='real',     # Shows 'u'
    overlay=None,
    mode='surface',       
    physical=True
)

HTML(ani.to_jshtml())

In [ ]:
plt.rcParams['animation.embed_limit'] = 2**128

# Visualize the 'imag' component, which corresponds to 'v' in the original 2-PDE system
ani = solver.animate(
    component='imag',     # Shows 'v'
    overlay=None,
    mode='surface',       
    physical=True
)

HTML(ani.to_jshtml())

In [ ]:
ani.save('coupled_2pde_complex.mp4', writer='ffmpeg', fps=20, dpi=100)
print("✅ Saved to coupled_2pde_complex.mp4")

print("\n" + "="*60)
print("OBSERVATIONS:")
print("="*60)
print("1. The initial pulse in 'u' splits into two waves moving")
print("   in opposite directions (left and right).")
print("2. If you animate the 'imag' component (which is 'v'),")
print("   you will see the waves in 'v' are phase-shifted")
print("   relative to 'u', proving the cross-coupling is working.")
print("3. The waves slowly decay due to the numerical")
print("   dissipation term (ν), keeping the solver stable.")
print("="*60)